# 02 — PlantVillage Disease Classification (Self-Contained Kaggle Training)

**AgriSense AI · Phase 2 (DEV_PLAN Tasks 2.1 – 2.12)**

This notebook trains, evaluates, calibrates, and exports the **Disease Detection Model** on Kaggle GPU.
It is **100% self-contained** (requires zero external repo cloning or missing module dependencies).

### Supported Experiments:
- **D-01:** Baseline 4-stage CNN (trained from scratch)
- **D-02:** ResNet-50 (ImageNet fine-tune)
- **D-03:** EfficientNet-B0 (Basic augmentation)
- **D-05:** EfficientNet-B0 (Strong augmentation: RandomRotation, ColorJitter, Cutout)
- **D-06:** **EfficientNet-B0 (Champion)** (Strong augmentation + Label Smoothing $\alpha = 0.10$)

> **Prerequisites in Kaggle:**
> 1. Set **Settings → Accelerator → GPU T4 x 2** (or **P100**)
> 2. Ensure dataset `abdallahalidev/plantvillage-dataset` is attached under `/kaggle/input/`

## 1. GPU & Environment Verification

In [ ]:
import os
import sys
import time
import json
import glob
import random
from pathlib import Path
from typing import Tuple, Dict, List

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# Set seed for full reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("*** WARNING: No GPU detected! Go to Settings -> Accelerator -> GPU. ***")

OUTPUT_DIR = Path("/kaggle/working/artifacts/disease_model") if Path("/kaggle/working").exists() else Path("artifacts/disease_model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Artifact output directory: {OUTPUT_DIR}")

## 2. Dataset Discovery & Split Loader

In [ ]:
# Find dataset location in Kaggle input or local workspace
def locate_plantvillage_root() -> Path:
    candidates = [
        Path("/kaggle/input/plantvillage-dataset/color"),
        Path("/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color"),
        Path("/kaggle/input/plantvillage-dataset/PlantVillage"),
        Path("/kaggle/input/plantvillage/PlantVillage"),
        Path("ml/data/raw/plantvillage/color"),
    ]
    for c in candidates:
        if c.is_dir():
            return c
    # Dynamic search for any directory containing Apple___Apple_scab
    search_roots = [Path("/kaggle/input"), Path("ml/data"), Path(".")]
    for s in search_roots:
        if s.exists():
            for match in s.rglob("*Apple___Apple_scab*"):
                if match.is_dir():
                    return match.parent
    raise FileNotFoundError("PlantVillage dataset not found. Please attach abdallahalidev/plantvillage-dataset.")

DATASET_ROOT = locate_plantvillage_root()
print(f"Found PlantVillage at: {DATASET_ROOT}")

# Function to standardize column names across different manifest formats
def standardize_split_df(d: pd.DataFrame) -> pd.DataFrame:
    d = d.copy()
    if "class_name" not in d.columns:
        for c in ["class_id", "condition_class", "label", "category"]:
            if c in d.columns:
                d["class_name"] = d[c]
                break
    if "filepath" not in d.columns:
        for c in ["file_path", "image_path", "path"]:
            if c in d.columns:
                d["filepath"] = d[c]
                break
    return d

# Check for split manifests or build leakage-aware stratified splits on the fly
split_candidates = [
    Path("/kaggle/working/artifacts/disease_data"),
    Path("ml/data/processed"),
]
train_csv = None
for sc in split_candidates:
    if (sc / "train_split.csv").exists():
        train_csv = sc / "train_split.csv"
        val_csv = sc / "val_split.csv"
        test_csv = sc / "test_split.csv"
        break

if train_csv and train_csv.exists():
    print(f"Loading existing Phase 1 splits from: {train_csv.parent}")
    train_df = standardize_split_df(pd.read_csv(train_csv))
    val_df = standardize_split_df(pd.read_csv(val_csv))
    test_df = standardize_split_df(pd.read_csv(test_csv))
else:
    print("Generating stratified 70/15/15 splits on the fly...")
    all_samples = []
    class_dirs = sorted([d for d in DATASET_ROOT.iterdir() if d.is_dir()])
    classes = [d.name for d in class_dirs]
    class_to_idx = {c: i for i, c in enumerate(classes)}
    
    for d in class_dirs:
        c_name = d.name
        for img_path in d.glob("*.JPG"):
            all_samples.append({"filepath": str(img_path.resolve()), "class_name": c_name, "class_idx": class_to_idx[c_name]})
        for img_path in d.glob("*.jpg"):
            all_samples.append({"filepath": str(img_path.resolve()), "class_name": c_name, "class_idx": class_to_idx[c_name]})
            
    full_df = pd.DataFrame(all_samples)
    # Stratified 70/15/15 split
    train_list, val_list, test_list = [], [], []
    for c_name, group in full_df.groupby("class_name"):
        shuffled = group.sample(frac=1.0, random_state=SEED)
        n = len(shuffled)
        n_train = int(0.70 * n)
        n_val = int(0.15 * n)
        train_list.append(shuffled.iloc[:n_train])
        val_list.append(shuffled.iloc[n_train:n_train+n_val])
        test_list.append(shuffled.iloc[n_train+n_val:])
        
    train_df = pd.concat(train_list, ignore_index=True)
    val_df = pd.concat(val_list, ignore_index=True)
    test_df = pd.concat(test_list, ignore_index=True)

# Sorted class names
classes = sorted(train_df["class_name"].astype(str).unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for i, c in enumerate(classes)}
num_classes = len(classes)

print(f"Splits ready: Train={len(train_df):,} | Val={len(val_df):,} | Test={len(test_df):,}")
print(f"Total unique classes: {num_classes}")

## 3. Dataset & DataLoaders with Augmentation Policies

In [ ]:
class SelfContainedPlantVillageDataset(Dataset):
    """Robust PyTorch Dataset with error handling for image paths across environments."""
    def __init__(self, df: pd.DataFrame, class_to_idx: Dict[str, int], transform=None, root_dir: Path = None):
        self.df = df.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.transform = transform
        self.root_dir = root_dir

    def __len__(self):
        return len(self.df)

    def _resolve_img_path(self, row: pd.Series) -> Path:
        # 1. Direct path check
        p = Path(str(row.get("filepath", "")))
        if p.exists():
            return p
        # 2. Check relative_path against root_dir
        if "relative_path" in row and self.root_dir:
            rp = self.root_dir / str(row["relative_path"])
            if rp.exists():
                return rp
        # 3. Check root_dir / class_name / filename
        if self.root_dir and "class_name" in row:
            cp = self.root_dir / str(row["class_name"]) / p.name
            if cp.exists():
                return cp
        return p

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self._resolve_img_path(row)
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception:
            # Fallback black image if missing/corrupt
            img = Image.new("RGB", (224, 224))
            
        if self.transform:
            img = self.transform(img)
            
        label = self.class_to_idx[str(row["class_name"])]
        return img, label

# ImageNet standard normalization
NORMALIZE = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

# Basic Augmentation
train_transform_basic = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    NORMALIZE,
])

# Strong Augmentation (Mitigating domain gap & background memorization)
train_transform_strong = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=25),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    NORMALIZE,
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.2), value="random"),
])

# Validation / Test deterministic transform
eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    NORMALIZE,
])

## 4. Model Architecture Builder

In [ ]:
class BaselineCNN(nn.Module):
    """D-01: 4-stage convolutional neural network trained from scratch."""
    def __init__(self, num_classes=38):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

def build_model(architecture: str, num_classes: int, pretrained: bool = True) -> nn.Module:
    arch = architecture.lower()
    if arch == "baseline_cnn":
        return BaselineCNN(num_classes=num_classes)
    elif arch == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        m = models.resnet50(weights=weights)
        in_f = m.fc.in_features
        m.fc = nn.Linear(in_f, num_classes)
        return m
    elif arch == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        m = models.efficientnet_b0(weights=weights)
        in_f = m.classifier[1].in_features
        m.classifier[1] = nn.Linear(in_f, num_classes)
        return m
    else:
        raise ValueError(f"Unsupported architecture: {arch}")

## 5. Evaluation & Calibration Metrics

In [ ]:
def calculate_ece(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15) -> float:
    """Compute Expected Calibration Error (ECE)."""
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = predictions == labels
    
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n_samples = len(labels)
    
    for i in range(n_bins):
        in_bin = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i + 1])
        prop_in_bin = np.mean(in_bin)
        if prop_in_bin > 0:
            accuracy_in_bin = np.mean(accuracies[in_bin])
            avg_confidence_in_bin = np.mean(confidences[in_bin])
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
            
    return float(ece)

@torch.no_grad()
def evaluate_dataset(model: nn.Module, loader: DataLoader, device: torch.device) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    model.eval()
    all_logits, all_labels = [], []
    for images, labels in loader:
        images = images.to(device)
        logits = model(images)
        all_logits.append(logits.cpu().numpy())
        all_labels.append(labels.numpy())
        
    logits = np.concatenate(all_logits, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    preds = np.argmax(logits, axis=1)
    
    exp_l = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    probs = exp_l / np.sum(exp_l, axis=1, keepdims=True)
    
    acc = float(accuracy_score(labels, preds))
    macro_f1 = float(f1_score(labels, preds, average="macro", zero_division=0))
    ece = calculate_ece(probs, labels)
    
    return {"accuracy": acc, "macro_f1": macro_f1, "ece": ece}, probs, labels

## 6. Select Configuration & Train

In [ ]:
# EXPERIMENT SELECTION (Recommended: 'D-06' for production champion)
EXPERIMENT = "D-06"

CONFIGS = {
    "D-01": {"arch": "baseline_cnn", "pretrained": False, "aug": "basic", "lr": 1e-3, "epochs": 10, "batch_size": 64, "smoothing": 0.0},
    "D-02": {"arch": "resnet50", "pretrained": True, "aug": "basic", "lr": 3e-4, "epochs": 10, "batch_size": 32, "smoothing": 0.0},
    "D-03": {"arch": "efficientnet_b0", "pretrained": True, "aug": "basic", "lr": 3e-4, "epochs": 10, "batch_size": 32, "smoothing": 0.0},
    "D-05": {"arch": "efficientnet_b0", "pretrained": True, "aug": "strong", "lr": 3e-4, "epochs": 10, "batch_size": 32, "smoothing": 0.0},
    "D-06": {"arch": "efficientnet_b0", "pretrained": True, "aug": "strong", "lr": 3e-4, "epochs": 10, "batch_size": 32, "smoothing": 0.1},
}

cfg = CONFIGS[EXPERIMENT]
print("Training with Configuration:", json.dumps(cfg, indent=2))

# Build DataLoaders
train_tf = train_transform_strong if cfg["aug"] == "strong" else train_transform_basic
train_ds = SelfContainedPlantVillageDataset(train_df, class_to_idx, transform=train_tf, root_dir=DATASET_ROOT)
val_ds = SelfContainedPlantVillageDataset(val_df, class_to_idx, transform=eval_transform, root_dir=DATASET_ROOT)
test_ds = SelfContainedPlantVillageDataset(test_df, class_to_idx, transform=eval_transform, root_dir=DATASET_ROOT)

train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=cfg["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=cfg["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

# Build Model, Loss, Optimizer
model = build_model(cfg["arch"], num_classes=num_classes, pretrained=cfg["pretrained"]).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=cfg["smoothing"])
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["epochs"], eta_min=cfg["lr"] * 0.01)
scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

best_val_f1 = 0.0
best_weights_path = OUTPUT_DIR / f"{EXPERIMENT}_best_model.pt"

print("\nStarting Training Loop...")
print("=" * 70)
for epoch in range(1, cfg["epochs"] + 1):
    t0 = time.time()
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
        
    scheduler.step()
    train_loss = total_loss / total
    train_acc = correct / total
    
    val_metrics, _, _ = evaluate_dataset(model, val_loader, device)
    elapsed = time.time() - t0
    
    print(f"Epoch [{epoch:02d}/{cfg['epochs']:02d}] ({elapsed:.1f}s) | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.1%} | "
          f"Val Acc: {val_metrics['accuracy']:.1%} Macro-F1: {val_metrics['macro_f1']:.4f}")
    
    if val_metrics["macro_f1"] > best_val_f1:
        best_val_f1 = val_metrics["macro_f1"]
        torch.save(model.state_dict(), best_weights_path)

print("=" * 70)
print(f"Training finished! Best Validation Macro-F1: {best_val_f1:.4f}")

## 7. Evaluate on Held-Out Test Set & Export Production Artifacts

In [ ]:
# Load best model weights
model.load_state_dict(torch.load(best_weights_path, map_location=device))
print(f"Loaded best weights from: {best_weights_path}")

test_metrics, test_probs, test_labels = evaluate_dataset(model, test_loader, device)
print("\n--- Held-Out Test Set Results ---")
print(f"Test Top-1 Accuracy: {test_metrics['accuracy']:.2%}")
print(f"Test Macro-F1 Score: {test_metrics['macro_f1']:.4f}")
print(f"Expected Calibration Error (ECE): {test_metrics['ece']:.4f}")

# Export to TorchScript
model.eval().to("cpu")
dummy_input = torch.randn(1, 3, 224, 224)
traced_model = torch.jit.trace(model, dummy_input)
export_pt_path = OUTPUT_DIR / "model.pt"
traced_model.save(str(export_pt_path))

# Verify Numerical Parity
with torch.no_grad():
    orig_logits = model(dummy_input).numpy()
    loaded_model = torch.jit.load(str(export_pt_path))
    traced_logits = loaded_model(dummy_input).numpy()
    max_error = float(np.max(np.abs(orig_logits - traced_logits)))

print(f"\nTorchScript exported to: {export_pt_path} ({export_pt_path.stat().st_size / 1024**2:.1f} MB)")
print(f"Parity Verified: {max_error <= 1e-4} (Max error: {max_error:.2e})")

# Save labels.json mapping
labels_dict = {str(i): c for i, c in enumerate(classes)}
with open(OUTPUT_DIR / "labels.json", "w", encoding="utf-8") as f:
    json.dump(labels_dict, f, indent=2)
print(f"Saved labels.json to: {OUTPUT_DIR / 'labels.json'}")
print("\nModel training, test evaluation, and export completed successfully!")